In [ ]:
!pip install --upgrade fastai

In [ ]:
from fastai.tabular.all import *
from geopy.distance import great_circle
path = Path('/kaggle/input/playground-series-s3e1')

def random_seed(seed_value): 
    np.random.seed(seed_value) 
    torch.manual_seed(seed_value) 
    random.seed(seed_value) 


random_seed(42)

# Introduction and aim of the notebook

In this notebook I'll show how easy and fast one could build a baseline in fastai using the [tabular_learner](https://docs.fast.ai/tabular.learner.html#tabular_learner).

The idea is to build something **quick and easy and then fine-tune it**. Skipping important steps such as EDA is never a good idea, however a step by step approach can often go a long way: **progress over perfection**!

Let's get started!

# Changelog

v3 - Add coast features

v2 - Add SaveModelCallback, ensembling 

v1 - Baseline

# Imports and Dataloaders

The first step is to import the training data and build the dataloader.

In [ ]:
train_df = pd.read_csv(path/'train.csv')
train_df.drop(columns=['id'], inplace=True)
print(train_df.shape)
train_df.head()

In [ ]:
# Add coast features based on https://www.kaggle.com/competitions/playground-series-s3e1/discussion/376542
coast = np.array([[32.664472968971786, -117.16139777220666],
         [33.20647603453836, -117.38308931734736],
         [33.77719697387153, -118.20238415808473],
         [34.46343131623148, -120.01447157053916],
         [35.42731619324845, -120.8819602254066],
         [35.9284107340049, -121.48920228383551],
         [36.982737132545495, -122.028973002425],
         [37.61147966825591, -122.49163361836126],
         [38.3559871217218, -123.06032062543764],
         [39.79260770260524, -123.82178288918176],
         [40.799744611668416, -124.18805587680554],
         [41.75588735544064, -124.19769463963775]])

def geodesic_dist(x, y):
    return great_circle(x, y).km

def coast_dist(df):
    for i in range(len(coast)):
        df[f'coast_{i}'] = df.apply(lambda x: geodesic_dist((x.Latitude, x.Longitude),coast[i]), axis=1)

    return df

train_df = coast_dist(train_df)

In [ ]:
test_df = pd.read_csv(path/'test.csv')
test_df = coast_dist(test_df)

Let's now pass this dataframe to the [TabularPandas](https://docs.fast.ai/tabular.core.html#tabularpandas) class of fastai, defining how we want to split our data (90% training, 10% validation) and that it's a regression task ([RegressionBlock](https://docs.fast.ai/data.block.html#regressionblock)):

In [ ]:
splits = RandomSplitter(valid_pct=0.15)(train_df)
tp = TabularPandas(train_df,
                   procs=[Normalize],
                   cont_names = list(train_df.columns[~train_df.columns.str.contains('MedHouseVal')]),
                   y_names='MedHouseVal',
                   y_block=RegressionBlock,
                   splits=splits)

The `procs` parameter helps us to apply a list of pre-processors to our data:

- `FillMissing` will fill the missing values in the continuous variables by the median of existing values
- `Normalize` will normalize the continuous variables (subtract the mean and divide by the std)

Let's check a couple of rows to see if everything looks ok:

In [ ]:
tp.xs.iloc[:2]

We can now build the dataloader and inspect a batch of data:

In [ ]:
dls = tp.dataloaders(bs=1024)
dls.show_batch()

# Model training

Now it's time to define the model, passing `rmse` as metric and fit for few epochs.

In [ ]:
learn = tabular_learner(dls, metrics=rmse)

In [ ]:
lrs = learn.lr_find(suggest_funcs=(minimum, steep, valley, slide))

Let's pick the value of `valley` as learning rate, using the SaveModelCallback to save the best model during training and load it at the end.

In [ ]:
learn.fit_one_cycle(10, lrs.valley, cbs=SaveModelCallback())

Let's check that we are using the best model found during training

In [ ]:
probs, target = learn.get_preds(dl=dls.valid)
rmse(probs, target)

Indeed the rmse is the same as the one found. 

Let's now predict on the test test by reading the csv and loading it using the `test_dl` method of the Dataloaders.

In [ ]:
dl = learn.dls.test_dl(test_df)

In [ ]:
preds, _ = learn.get_preds(dl=dl)
preds

# Ensembling

We can scale up a little bit by running n times the above workflow, each trained from different random starting points and/or different model architecture, and average the predictions. This would improve our ability to generalize better, reducing overfitting.

In [ ]:
def train(name, layers):
    splits = RandomSplitter(valid_pct=0.15)(train_df) # There's no seed set, so each time we have different training data
    tp = TabularPandas(train_df,
                       procs=[Normalize],
                       cont_names = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
                                     'Population', 'AveOccup', 'Latitude', 'Longitude'],
                       y_names='MedHouseVal',
                       y_block=RegressionBlock,
                       splits=splits)
    dls = tp.dataloaders(bs=1024)
    learn = tabular_learner(dls, metrics=rmse)
    lrs = learn.lr_find(suggest_funcs=(minimum, steep, valley, slide), show_plot=False)
    
    with learn.no_bar(),learn.no_logging():
        learn.fit_one_cycle(15, lrs.valley, cbs=SaveModelCallback(fname=name))
    
    dl = learn.dls.test_dl(test_df)
    preds, _ = learn.get_preds(dl=dl)
    return preds

In [ ]:
architectures = [
    ('200-100', [200, 100]),
    ('150-125', [150, 125]),
    ('250-150', [250, 150]),
    ('200-100_v2', [200, 100]),
    ('150-125_v2', [150, 125]),
    ('250-150_v2', [250, 150])
]
predictions = [train(n, i) for n,i in architectures]

Let's check the models folder where all our best models have been saved.

In [ ]:
(Path('/kaggle/working/')/'models').ls()

Let's stack the predictions, taking the average.

In [ ]:
ensemble_predictions = torch.stack(predictions).mean(0)
ensemble_predictions

# Submission

Finally, let's create our submission to the competition.

In [ ]:
submission = pd.read_csv(path/'sample_submission.csv')
submission

In [ ]:
submission['MedHouseVal'] = ensemble_predictions
submission.head()

In [ ]:
submission.to_csv('submission.csv', index=False)

# Next steps

As a next steps I will try to ensemble more models and new features.

Stay tuned!